In [ ]:
import pandas as pd
import numpy as np

import json

import re

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import nltk
nltk.download()

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from rake_nltk import Rake

In [ ]:
FILEPATH="../data/cleaned/"

df = pd.read_csv(FILEPATH+"movie_data_cleaned.csv")

In [ ]:
df.head()

In [ ]:
df.loc[0, 'genres']

In [ ]:
json.loads(df.loc[0, 'genres'])

# Data Extraction

### Extracting genres and keywords and storing into dataset

In [ ]:
def data_extraction(x):
    data_list = []
    x = json.loads(x)
    for i, item in enumerate(x):
        data_list.append(x[i]['name'])
    final_data = ', '.join(data_list)
    return final_data

In [ ]:
df.loc[:, 'genres'] = df.loc[:, 'genres'].apply(lambda x: data_extraction(x))
df.loc[:, 'keywords'] = df.loc[:, 'keywords'].apply(lambda x: data_extraction(x))

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

# Checking Available Genres:

In [ ]:
x = df.loc[0,'genres']
x.split(', ')

In [ ]:
genres = set()
for i in df['genres']:
    i = i.split(', ')
    genres.update(i)

In [ ]:
genres

In [ ]:
genres.remove('')

In [ ]:
genres

In [ ]:
df['overview'] = df['overview'].fillna(value='')

In [ ]:
df['overview'].isnull().sum() # no null values present

In [ ]:
description = df.loc[0,'overview']
description

In [ ]:
# customize stopwords according to your requirement
cust_stopwords = set(stopwords.words('english'))

### We are using Rake algorithm for quick text extraction

In [ ]:
def text_extraction(x):
    r = Rake(stopwords=cust_stopwords)
    r.extract_keywords_from_text(x)
    keyword_score = r.get_word_degrees()
    res = ' '.join(list(keyword_score.keys()))
    return res

In [ ]:
df.loc[:, 'overview'] = df.loc[:, 'overview'].apply(lambda x: text_extraction(x))

In [ ]:
description = df.loc[0,'overview']
description

In [ ]:
df.head()

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
for item in df.iterrows():
    print(item)

In [ ]:
for index, row in df.iterrows():
    print("{}\n\n{}\n".format(index, row))

``` python
df['meta_tags'] = '' # creating an empty column


for index, row in df.iterrows():
    #meta tags
    genre = row['genres'].replace(',', '').lower()
    keyword = row['keywords'].replace(',', '').lower()
    overview = row['overview'].lower()
    overview_filtered = re.sub(pattern="[^a-z0-9 ]+", repl="", string=overview)

    # combine all features in to meta tags - string concatination
    combined_keywords = genre+' '+keyword+' '+overview_filtered

    df.at[index, 'meta_tags'] = combined_keywords
```

### Creating weighted meta data

In [ ]:
# clean columns
df['genres_clean'] = df['genres'].str.replace(',', '', regex=False).str.lower()
df['keywords_clean'] = df['keywords'].str.replace(',', '', regex=False).str.lower()

df['overview_clean'] = (
    df['overview']
    .str.lower()
    .str.replace(r'[^a-z0-9 ]+', '', regex=True)
)

# combine into meta_tags
df['meta_tags'] = (
    df['genres_clean'] + ' ' +
    df['keywords_clean'] + ' ' +
    df['overview_clean']
)

In [ ]:
df.head()

In [ ]:
df.loc[0,'meta_tags']

In [ ]:
df_meta = df[['title', 'meta_tags']]

### We are using lemmatization as lexicon normalization instead of using stemming because meta data is sensitive and needs to be clipped carefully without changing the meaning.

In [ ]:
def lemmatizer_word(x):
    lemm_obj = WordNetLemmatizer()
    word_tokens = nltk.word_tokenize(text=x, language='english', preserve_line=False)
    lemm_words = []
    for i in word_tokens:
        lemm_words.append(lemm_obj.lemmatize(i))
    res = ' '.join(lemm_words)
    return res

In [ ]:
df_meta.loc[:, 'meta_tags'] = df_meta.loc[:, 'meta_tags'].apply(lambda x: lemmatizer_word(x))

In [ ]:
df_meta.loc[0, 'meta_tags']

### Exporting file

In [ ]:
EXPORT_FILEPATH="../data/train/"

df_meta.to_csv(EXPORT_FILEPATH+"meta_data.csv", index=False)